# Suite2p — batch processing

Load one settings file, then run the same settings on several recordings.

* `settings`: **how** to process (shared by all recordings).
* `db`: **what** to process (one per recording: data and save paths, planes, channels).

In [ ]:
import json
import logging
import shutil
from pathlib import Path
import suite2p
from suite2p import default_db
from suite2p.run_s2p import logger_setup, get_save_folder

## 1. Load the settings

In [ ]:
settings_path = r"C:\Users\User\Documents\Code\suite2p\Python_demo_Federico\Dendrites_iGluSNFR4s_settings.json"

with open(settings_path) as f:
    settings = json.load(f)

## 2. One `db` per recording

In [ ]:
dbs = [
    {
        "data_path":  [
            r"Z:\Lab_Share\Federico\suite2p_samples\AL042\1",
            r"Z:\Lab_Share\Federico\suite2p_samples\AL042\2",
        ],
        "save_path0": r"Z:\Lab_Share\Federico\suite2p_samples\processed\AL042",
    },
 
    {
        "data_path":  [r"Z:\Lab_Share\Federico\suite2p_samples\NM035\1"],
        "save_path0": r"Z:\Lab_Share\Federico\suite2p_samples\processed\NM035",
    },
]

## 3. Run

If a recording already has results, suite2p reuses the old `reg_outputs.npy` and **skips registration**, even when it has just rebuilt an unregistered binary from the TIFFs. To avoid processing an unaligned movie:

* `OVERWRITE = False`: recordings that already have results are skipped.
* `OVERWRITE = True`: that recording's `suite2p` folder is deleted and processed from scratch.

In [ ]:
OVERWRITE = False

s2p_logger = logging.getLogger("suite2p")

for db in dbs:
    db = {**default_db(), **db}
    save_folder = Path(get_save_folder(db))

    if any(save_folder.glob("plane*/reg_outputs.npy")):
        if not OVERWRITE:
            print(f"SKIPPING (already processed, set OVERWRITE = True to redo): {save_folder}")
            continue
        print(f"Deleting previous results: {save_folder}")
        shutil.rmtree(save_folder)

    # run_s2p() doesn't set up logging itself (the CLI/GUI do), so without this
    # the per-stage messages are not shown. Drop the previous recording's log
    # file handler so each run.log only contains its own recording.
    for h in [h for h in s2p_logger.handlers if isinstance(h, logging.FileHandler)]:
        s2p_logger.removeHandler(h)
        h.close()
    logger_setup(str(save_folder))   # prints to this cell + writes <save_folder>/run.log

    suite2p.run_s2p(settings=settings, db=db)